### Create Unity Catalog Structure

- Create Catalog: olist
- Create Schemas: bronze, silver, gold, landing
- Create Volume : Tied to landing folder in olist container -> To be easily able to access raw files using file explorer like paths (instead of abfss URL) and UI

_External Location already created using GUI_


In [0]:
%run "../shared/00.common-variables"

In [0]:
show external locations

In [0]:
CREATE CATALOG IF NOT EXISTS olist
   MANAGED LOCATION 'abfss://olist@olistretailplatform.dfs.core.windows.net/'
   COMMENT 'This is the main Catalog for olist retail lakehouse project'

In [0]:
CREATE SCHEMA IF NOT EXISTS olist.landing; -- Not defining any managed location/path here as we don't want to create any tables in landing schema. it will only contain the raw data files. we'll create tables in bronze, gold and silver Schemas.
CREATE SCHEMA IF NOT EXISTS olist.bronze
    MANAGED LOCATION 'abfss://olist@olistretailplatform.dfs.core.windows.net/bronze';
CREATE SCHEMA IF NOT EXISTS olist.silver
    MANAGED LOCATION 'abfss://olist@olistretailplatform.dfs.core.windows.net/silver';
CREATE SCHEMA IF NOT EXISTS olist.gold
    MANAGED LOCATION 'abfss://olist@olistretailplatform.dfs.core.windows.net/gold';

In [0]:
%python
spark.sql(f"""CREATE TABLE IF NOT EXISTS olist.bronze.orders
(
    order_id                         STRING,
    order_status                     STRING,
    customer_id                      STRING,
    order_purchase_timestamp         TIMESTAMP,
    order_approved_at                TIMESTAMP,
    order_delivered_carrier_date     TIMESTAMP,
    order_delivered_customer_date    TIMESTAMP,
    order_estimated_delivery_date    DATE,
    _corrupt_record                  STRING,
    ingestion_timestamp              TIMESTAMP NOT NULL,
    source_file_name                 STRING NOT NULL,
    source_system                    STRING NOT NULL,
    pipeline_run_id                  STRING NOT NULL
)
USING DELTA
LOCATION "abfss://olist@olistretailplatform.dfs.core.windows.net/bronze/orders"
""")

In [0]:
%python
spark.sql(f"""CREATE TABLE IF NOT EXISTS olist.bronze.customers 
(
    customer_id                STRING,
    customer_unique_id         STRING,
    customer_zip_code_prefix   STRING,
    customer_city              STRING,
    customer_state             STRING,
    _corrupt_record            STRING,
    ingestion_timestamp        TIMESTAMP NOT NULL,
    source_file_name           STRING NOT NULL,
    source_system              STRING NOT NULL,
    pipeline_run_id            STRING NOT NULL
)
USING DELTA
LOCATION "abfss://olist@olistretailplatform.dfs.core.windows.net/bronze/customers"
""")

In [0]:
%python
spark.sql(f"""CREATE TABLE IF NOT EXISTS olist.bronze.customers
(
    product_id                  STRING,
    product_category_name       STRING,
    product_name_lenght         INTEGER,
    product_description_lenght  INTEGER,
    product_photos_qty          INTEGER,
    product_weight_g            INTEGER,
    product_length_cm           INTEGER,
    product_height_cm           INTEGER,
    product_width_cm            INTEGER,

    _corrupt_record            STRING,
    ingestion_timestamp        TIMESTAMP NOT NULL,
    source_file_name           STRING NOT NULL,
    source_system              STRING NOT NULL,
    pipeline_run_id            STRING NOT NULL
)
USING DELTA
LOCATION "abfss://olist@olistretailplatform.dfs.core.windows.net/bronze/products"
""")

In [0]:
%python
spark.sql(f"""CREATE TABLE IF NOT EXISTS olist.bronze.sellers 
(
    seller_id                   STRING,
    seller_zip_code_prefix      INTEGER,
    seller_city                 STRING,
    seller_state                STRING,
    _corrupt_record             STRING,
    ingestion_timestamp         TIMESTAMP NOT NULL,
    source_file_name            STRING NOT NULL,
    source_system               STRING NOT NULL,
    pipeline_run_id             STRING NOT NULL
)
USING DELTA
LOCATION "abfss://olist@olistretailplatform.dfs.core.windows.net/bronze/sellers"
""")

In [0]:
%python
spark.sql(f"""CREATE TABLE IF NOT EXISTS olist.bronze.payments
(
    order_id                   STRING,
    payment_sequential         INTEGER,
    payment_type               STRING,
    payment_installments       INTEGER,
    payment_value              FLOAT,
    _corrupt_record            STRING,
    ingestion_timestamp        TIMESTAMP NOT NULL,
    source_file_name           STRING NOT NULL,
    source_system              STRING NOT NULL,
    pipeline_run_id            STRING NOT NULL
)
USING DELTA
LOCATION "abfss://olist@olistretailplatform.dfs.core.windows.net/bronze/payments"
""")

In [0]:
%python
spark.sql(f"""CREATE TABLE IF NOT EXISTS olist.bronze.reviews
(
    review_id                  STRING,
    order_id                   STRING,
    review_score               INTEGER,
    review_comment_title       STRING,
    review_comment_message     STRING,
    review_creation_date       DATE,
    review_answer_timestamp    TIMESTAMP,
    _corrupt_record            STRING,
    ingestion_timestamp        TIMESTAMP NOT NULL,
    source_file_name           STRING NOT NULL,
    source_system              STRING NOT NULL,
    pipeline_run_id            STRING NOT NULL
)
USING DELTA
LOCATION "abfss://olist@olistretailplatform.dfs.core.windows.net/bronze/reviews"
""")

In [0]:
%python
spark.sql(f"""CREATE TABLE IF NOT EXISTS olist.bronze.order_items
(
    order_id                  STRING,
    order_item_id             INTEGER,
    product_id                STRING,
    seller_id                 STRING,
    shipping_limit_date       TIMESTAMP,
    price                     FLOAT,
    freight_value             FLOAT,
    
    _corrupt_record           STRING,
    ingestion_timestamp       TIMESTAMP NOT NULL,
    source_file_name          STRING NOT NULL,
    source_system             STRING NOT NULL,
    pipeline_run_id           STRING NOT NULL
)
USING DELTA
LOCATION "abfss://olist@olistretailplatform.dfs.core.windows.net/bronze/order_items"
""")

In [0]:
%python
spark.sql(f"""CREATE TABLE IF NOT EXISTS olist.bronze.geolocations 
(
    geolocation_zip_code_prefix    STRING,
    geolocation_lat                DOUBLE,
    geolocation_lng                DOUBLE,
    geolocation_city               STRING,
    geolocation_state              STRING,

    _corrupt_record                STRING,
    ingestion_timestamp            TIMESTAMP NOT NULL,
    source_file_name               STRING NOT NULL,
    source_system                  STRING NOT NULL,
    pipeline_run_id                STRING NOT NULL
)
USING DELTA
LOCATION "abfss://olist@olistretailplatform.dfs.core.windows.net/bronze/geolocations"
""")

### Create Volume Files

**Why create volumes? Since we are already able to access files using abfss protocol from our external location?**

1. Issue: 
    - With just using the External Location ==> we need to use the URL every time which is tedious

2. Solution (provided by Volumes):
    - Volumes wraps the external location to expose the storage as a MANAGED LOCATION in databricks.
    - Provides a simple FILE EXPLORERE-like UI and easy access (simple file path conventions eg: **/Volumes/formula1/landing/files**) to files w/o having to use the abfss-URL every time
    - We can easily upload data to volume using DBricks UI
    - We can set permission for Access Control on the Volume
    



In [0]:
%python
dbutils.fs.ls("abfss://olist@olistretailplatform.dfs.core.windows.net/")

In [0]:
CREATE EXTERNAL VOLUME IF NOT EXISTS olist.landing.files
LOCATION 'abfss://olist@olistretailplatform.dfs.core.windows.net/landing'

In [0]:
%python
dbutils.fs.ls("/Volumes/olist/landing/files")

In [0]:
CREATE SCHEMA IF NOT EXISTS olist.logs

In [0]:
%python
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog_name}.{logs_schema}.pipeline_logs
(
    pipeline_run_id           STRING NOT NULL,
    pipeline_name             STRING NOT NULL,
    table_name                STRING NOT NULL,
    start_time                TIMESTAMP,
    end_time                  TIMESTAMP,
    status                    STRING,
    source_row_count          INT,
    target_row_count          INT,
    null_business_key_count       INT,
    duplicated_business_key_count INT,
    corrupt_row_count         INT,
    error_message             STRING
)
USING DELTA
LOCATION "abfss://olist@olistretailplatform.dfs.core.windows.net/logs/pipeline_logs"
""")

In [0]:
%python
spark.sql(f"""CREATE TABLE IF NOT EXISTS {catalog_name}.{logs_schema}.pipeline_control
(
    file_checksum         STRING NOT NULL,
    file_name             STRING NOT NULL,
    file_path             STRING NOT NULL,
    
    pipeline_name         STRING NOT NULL,
    pipeline_run_id       STRING NOT NULL,
    table_name            STRING NOT NULL,
    
    status                STRING NOT NULL,
    retry_count           INT NOT NULL,
    start_time            TIMESTAMP,
    end_time              TIMESTAMP,
    source_row_count      INT,
    target_row_count      INT,
    error_message         STRING,
    file_location         STRING NOT NULL
)
USING DELTA
LOCATION "abfss://olist@olistretailplatform.dfs.core.windows.net/logs/pipeline_control"
""")

#### Demo

In [0]:
CREATE SCHEMA IF NOT EXISTS olist.demo

In [0]:
CREATE EXTERNAL VOLUME IF NOT EXISTS olist.demo.files
LOCATION 'abfss://olist@olistretailplatform.dfs.core.windows.net/demo'